# Entering the Discrete Deformation Calculus

This notebook measures a state, replays lawful tapes and reads their receipts. Every number is computed exactly over the rationals by the package code.

Prerequisites: rational arithmetic and directed graphs.
Objectives: compute both diagnostics; distinguish abstract law checks from measured replay; identify the witness or hypothesis that a rejected step lacks.


In [1]:
import json
import sys
from pathlib import Path

# Locate the package from wherever the kernel starts, then import the flat modules.
here = Path.cwd().resolve()
candidates = (here, *here.parents, here / "omega_engine_atlas/coda_atlas/omega_engine_discrete_deformation_calculus")
root = next(p for p in candidates if (p / "discrete_deformation_calculus.md").is_file())
sys.path.insert(0, str(root / "code" / "ddc"))
metadata = json.loads((root / "zenodo_metadata.json").read_text(encoding="utf-8"))
print("Discrete Deformation Calculus", metadata["version"])

Discrete Deformation Calculus 1.0.3


## 1. A measured state

Φ₁ is the ℓ¹ distance to the image of d₀; Γ₂ is the face-closure defect.

In [2]:
import cochain
from automaton import measure

square = cochain.Complex(("a", "b", "c", "d"), (("a", "b"), ("b", "c"), ("c", "d"), ("d", "a")))
state = measure(square, (1, 0, 0, 0), "square")
print(state.phi, state.gamma, state.cls)
assert (state.phi, state.gamma, state.cls) == (1, 0, "ClosedResidual")

1 0 ClosedResidual


## 2. Lawful tapes and their receipts

In [3]:
import demos

for demo in demos.all_demos():
    if demo.lawful:
        print(f"{demo.name:32s} {demo.letters:7s} {demo.run.final_control:13s} {demo.run.tape.latest_hash[:16]}")

demo1_exact_CRET                 CRET    Terminalized  7b3e9a78f0eb9787
demo1_open_CRM                   CRM     Refused       ecea759d280c38f0
demo2_CRM                        CRM     Refused       d5448cd3cad2fa46
demo3_CRT_residual_terminal      CRT     Terminalized  03363c0b1b43cf87
demo3_CRLT_licensed              CRΛT    Terminalized  bd71c33fbc160100
demo3_CRMT_requires_lift         CRMT    Terminalized  fa7b9d0b3cd7be2d
demo4_CM                         CM      Refused       7eb8fdf516cf0618
demo4_CLT_detector_lift          CΛT     Terminalized  286bd32777c930aa
demo5_DIT                        DIT     Terminalized  1e70ab2c31485023
demo5_MT_refusal_record          MT      Terminalized  a0d78fd6f7c90788
newton_leibniz_CREIT             CREIT   Terminalized  0f17b5d15b14de11
credit                           CREDIT  Terminalized  cc6c764943b5843b


## 3. Newton–Leibniz as a tape

The left Riemann sum plus the exact correction telescopes to F(10) − F(0).

In [4]:
demo, numbers = demos.newton_leibniz()
print({k: str(v) for k, v in numbers.items()}, demo.letters, demo.lawful)
assert numbers["left_sum"] + numbers["defect"] == numbers["integral"] == numbers["telescoped"] == 50

{'left_sum': '45', 'defect': '5', 'integral': '50', 'telescoped': '50'} CREIT True


## 4. CREDIT

A lawful C·R·E·D·I·T requires an exact start and a booked descent. All six step laws must also hold, including the I receipt.

In [5]:
print(demos.credit().lawful, demos.credit(initial_defect=(1, 1, 1)).run.violations,
      demos.credit(ledger=False).run.violations)
assert demos.credit().lawful
assert not demos.credit(initial_defect=(1, 1, 1)).lawful
assert "MissingLossLedger" in demos.credit(ledger=False).run.violations


True ('ExactRepairFromNonExactClass',) ('MissingLossLedger',)


## 5. A representative norm is not the quotient minimum

Before executing: on increasingly oriented K4, put −1 on all six edges and choose the forest rooted at a. Predict the register norm and check Φ₁ with both primal and dual witnesses. R preserves the quotient even when its representative is not a minimizer.

In [6]:
from itertools import combinations
from residual_projection import GraphResidualProjector

vertices = tuple("abcd")
edges = tuple(combinations(vertices, 2))
k4 = cochain.Complex(vertices, edges)
x = (-1,) * 6
receipt = GraphResidualProjector(list(vertices), list(edges)).project(dict(zip(edges, x)))
cert = cochain.phi1_certificate(k4, x)
print("register:", receipt.cycle_coordinates, "recorded norm:", receipt.residual_norm, "Phi1:", cert.value)
assert receipt.residual_norm == 3 and cert.value == 2
assert cochain.check_certificate(k4, x, cert)

register: (Fraction(-1, 1), Fraction(-1, 1), Fraction(-1, 1)) recorded norm: 3 Phi1: 2


## 6. E repairs raw defect without lowering the diagnostics

On a path every cochain is exact. Predict the raw norm and both diagnostics for (2, −3), then repair with potential (0, 2, −1). Explain why this is compatible with the no-free-decrease theorem.

In [7]:
path = cochain.Complex(tuple("abc"), (("a", "b"), ("b", "c")))
raw = (2, -3)
potential = {"a": 0, "b": 2, "c": -1}
repaired = tuple(a - b for a, b in zip(raw, path.d0(potential)))
print("raw norms:", cochain.l1(raw), cochain.l1(repaired), "diagnostics:",
      cochain.phi1(path, raw), cochain.gamma2(path, raw))
assert repaired == (0, 0)
assert cochain.phi1(path, raw) == cochain.gamma2(path, raw) == 0

raw norms: 5 0 diagnostics: 0 0


## Exercise and feedback

Double the hollow-square defect: Φ₁ doubles and Γ₂ stays zero. Add the square's signed face: Γ₂ becomes positive, while d₀ and Φ₁ stay the same. For K4, verify the potential (0, −1, −1, −2) and flow (−1, 0, 1, −1, 0, −1) directly.

A closed cochain need not be exact; a forest register norm need not be minimal; E can reduce raw norm when Φ₁ was already zero. See the [study guide](../docs/study_guide.md) for the evidence boundaries.